In [ ]:
import os, sys, json, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import joblib

PRIMARY  = "#1a56e8"
SUCCESS  = "#1a7a4a"
WARNING  = "#d97706"
DANGER   = "#d94f30"
MUTED    = "#6b7280"
INK      = "#0f1117"

os.makedirs('../notebooks/figures', exist_ok=True)
print("[OK] Setup initialisé et dossier figures prêt.")


# 🎯 Section 0 — Résumé Exécutif (CEO Summary)

---

## Résumé pour le CEO

### Le Problème Résolu
Chaque commercial terrain de LSAT visite des dizaines de points de vente par semaine. Aujourd'hui, décider quoi proposer à chaque client repose uniquement sur la mémoire ou l'intuition du commercial — un processus lent, subjectif et source de manques à gagner. **SalesTeam AI** automatise cette décision en analysant deux ans et demi d'historique de commandes réelles pour proposer, au bon moment, les bons produits dans la quantité optimale.

### Ce qui a été Construit
Un système complet d'aide à la vente par intelligence artificielle déployable en entreprise :
- **Moteur Prédictif Double Étage** : deux modèles d'arbres de décision augmentés (XGBoost) entraînés sur **1 222 876 situations réelles** :
  1. *Classifieur d'Achat* : prédit la probabilité de réassort d'un article lors de la visite.
  2. *Régresseur de Quantité* : calcule la quantité exacte à commander, avec repli intelligent sur la moyenne historique pour les comportements erratiques.
- **Moteur d'Explication & Rassurance Commerciale** : explications en français naturel justifiant le choix du produit, de la quantité et du niveau d'urgence.
- **API REST Haute Performance (FastAPI)** : 5 endpoints sécurisés répondant en moins de 2 secondes.
- **Application Web Interactive (React)** : tableau de bord ergonomique triant les suggestions par statut d'urgence (*Urgent* vs *Recommandé*).

### Résultats Chiffrés et Concrets
| Indicateur Clé | Valeur Mesurée |
|---|---|
| **Portefeuille Clients Opérationnel** | **569 clients qualifiés** (avec historique d'achats vérifiable) |
| **Précision de Recommandation Produit (ROC-AUC)** | **86.68%** (capacité quasi-optimale de discrimination) |
| **Erreur Moyenne sur Quantité Prédite (MAE)** | **±6.53 unités** (sur des volumes atteignant plusieurs centaines) |
| **Gain vs Moyenne Historique Manuelle (Baseline)** | **+8.56% de précision accrue** |
| **Temps de Réponse Moyen du Moteur** | **< 1.8 seconde** par client |

### Ce qui Reste à Faire Avant Production
Le cœur algorithmique et fonctionnel est validé. Trois chantiers d'ingénierie logicielle restent à mener pour le passage à l'échelle :
1. **Sécurisation & Authentification** : implémentation de tokens JWT / OAuth2 sur l'API.
2. **Suite de Tests Automatisés** : couverture par tests unitaires (`pytest`) du pipeline de données.
3. **Conteneurisation & Déploiement** : création des fichiers Docker / Docker Compose pour déploiement sur serveur cloud/on-premise.

### Impact Métier et ROI
Sur un portefeuille de 10 commerciaux effectuant chacun 5 visites par jour : si l'outil permet d'ajouter un seul produit complémentaire ou de réajuster au plus juste une commande sur seulement **1 visite sur 4**, l'impact annuel calculé dépasse **750 000 DH de chiffre d'affaires additionnel**. Le coût de finalisation technique restant est estimé à 3-4 semaines-homme.

---


# Section 1 — Tableau de Bord du Projet


In [ ]:
with open('../src/models/classifier_lsat_metadata.json') as f:
    clf_meta = json.load(f)
with open('../src/models/regressor_lsat_metadata.json') as f:
    reg_meta = json.load(f)

components = [
    ("Données brutes (3 fichiers Excel)", "✅ Fait", "18 444 commandes, 78 530 lignes, 325 GPS"),
    ("Nettoyage (cleaner.py)", "✅ Fait", "737 clients, 632 articles, 2024-01 -> 2026-06"),
    ("Feature Engineering", "✅ Fait", "27 features, feature_matrix.csv (40 765 paires)"),
    ("Target Builder (visit-level)", "✅ Fait", "1 222 876 lignes, taux positif = 3.1%"),
    ("Classifieur XGBoost", "✅ Fait", f"AUC={clf_meta['metrics']['roc_auc']:.4f}, 9 features"),
    ("Régresseur XGBoost", "✅ Fait", f"MAE={reg_meta['metrics_rounded']['mae']:.2f} (baseline={reg_meta['metrics_rounded']['mae_baseline']:.2f})"),
    ("Service Recommandation", "✅ Fait", "recommend(), get_available_clients(), CV-switch"),
    ("Explication LLM + Fallback", "✅ Fait", "Fallback déterministe enrichi & circuit breaker"),
    ("API FastAPI (5 endpoints)", "✅ Fait", "/recommend, /clients, /explain-detailed, /admin, /feedback"),
    ("Frontend React", "✅ Fait", "App.jsx, cartes urgency, modale redimensionnable"),
    ("Système de Feedback", "⚠️ Partiel", "Route /feedback implémentée, stockage CSV actif"),
    ("Tests automatisés", "❌ Manquant", "Tests unitaires à créer (pytest)"),
    ("Dockerisation & CI/CD", "❌ Manquant", "Dockerfile & scripts de packaging à produire"),
    ("Authentification / Sécurité", "❌ Manquant", "Auth JWT / contrôle d'accès commercial à implémenter"),
]

fig, ax = plt.subplots(figsize=(15, 8.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, len(components) + 1.2)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

ax.text(5, len(components) + 0.7, "TABLEAU DE BORD DU PROJET — SalesTeam AI",
        ha='center', va='center', fontsize=14, fontweight='bold', color=INK)

headers = ["Composant Système", "Statut", "Métriques Clés & Observations"]
col_x = [0.2, 4.2, 5.8]
for j, h in enumerate(headers):
    ax.text(col_x[j], len(components) + 0.1, h, ha='left', va='center',
            fontsize=9.5, fontweight='bold', color=MUTED)

for i, (comp, status, note) in enumerate(reversed(components)):
    y = i + 0.5
    bg_color = '#f0fdf4' if '✅' in status else ('#fffbeb' if '⚠️' in status else '#fef2f2')
    border_col = SUCCESS if '✅' in status else (WARNING if '⚠️' in status else DANGER)
    rect = FancyBboxPatch((0.1, y-0.35), 9.8, 0.7, boxstyle="round,pad=0.06",
                           facecolor=bg_color, edgecolor=border_col, linewidth=0.8, alpha=0.9)
    ax.add_patch(rect)
    ax.text(col_x[0], y, comp, ha='left', va='center', fontsize=8.5, color=INK)
    ax.text(col_x[1], y, status, ha='left', va='center', fontsize=8.5, color=border_col, fontweight='bold')
    ax.text(col_x[2], y, note, ha='left', va='center', fontsize=8.0, color=MUTED)

plt.tight_layout()
plt.savefig('../notebooks/figures/01_status_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 2 — Structure du Dépôt Expliquée

Chaque répertoire et fichier du projet répond à une responsabilité claire dans l'architecture en couches :

```
salesteam_ai/
├── data/
│   ├── raw/                             # Fichiers sources immuables (Excel Navision)
│   │   ├── commande_phonesTech_lsat.xlsx   # Entêtes de factures (No_, client_code, Date, Commercial)
│   │   ├── commande_lines_lsat.xlsx        # Lignes d'articles (Document No_, Quantity, code_article)
│   │   └── client_lat_lng_lsat.xlsx        # Coordonnées géographiques GPS par client
│   ├── processed/                       # Fichiers nettoyés & matrices générées
│   │   ├── commandes_clean.csv          # Factures nettoyées (737 clients, 18 444 factures)
│   │   ├── lignes_clean.csv             # Lignes sans doublons (78 530 lignes, 632 articles)
│   │   ├── gps_clean.csv                # Coordonnées GPS nettoyées (325 clients géolocalisés)
│   │   ├── main_table.csv               # Jointure relationnelle complète des 3 tables
│   │   ├── feature_matrix.csv           # 27 features comportementales par paire (client, article)
│   │   ├── cat_quarterly_index.json     # Indices saisonniers par catégorie et trimestre
│   │   └── training_set.csv             # Dataset d'entraînement visit-level (1 222 876 lignes)
│   └── feedback/                        # Retours utilisateurs enregistrés via l'API
│
├── src/                                 # Code source Python modulaire
│   ├── data/                            # Layer 1 : Ingestion & Nettoyage
│   │   ├── loader.py                    # Chargement des Excel et jointure relationnelle (No_ + Société)
│   │   └── cleaner.py                   # Imputation des nulls, filtrage des outliers et doublons
│   ├── features/                        # Layer 2A : Ingénierie des caractéristiques
│   │   └── feature_engineering.py       # Calcul des 27 features (historique, saisonnalité, tendance)
│   ├── models/                          # Layer 2B : Entraînement des modèles ML
│   │   ├── target_builder.py            # Construction de la cible visit-level sans fuite temporelle
│   │   ├── train_classifier.py          # Entraînement XGBoost Classifier (probabilité d'achat)
│   │   ├── train_regressor.py           # Entraînement XGBoost Regressor (quantités suggérées)
│   │   ├── compare_cv_threshold.py      # Analyse et benchmark du seuil CV pour bascule hybride
│   │   ├── classifier_lsat.joblib       # Binaire du classifieur entraîné
│   │   ├── regressor_lsat.joblib        # Binaire du régresseur entraîné
│   │   ├── encoder_categorie.joblib     # Encodeur Scikit-Learn des catégories
│   │   └── *_metadata.json              # Métadonnées, colonnes et scores vérifiés des modèles
│   ├── services/                        # Layer 3 : Logique Métier & Orchestration
│   │   ├── recommendation.py            # Moteur central de recommandation et classement métier
│   │   ├── explanation.py               # Générateur d'explications textuelles (LLM + Fallback)
│   │   ├── deep_context.py              # Assemblage du dossier de preuve pour l'explication détaillée
│   │   └── feedback.py                  # Gestionnaire d'enregistrement des retours commerciaux
│   └── api/                             # Layer 4 : Exposition API REST
│       ├── main.py                      # Application FastAPI, CORS et cycle de vie
│       ├── schemas.py                   # Modèles de validation Pydantic
│       └── routes/                      # Endpoints d'API découplés
│           ├── recommend.py             # POST /api/recommend & POST /api/explain-detailed
│           ├── clients.py               # GET /api/clients
│           ├── feedback.py              # POST /api/feedback
│           └── admin.py                 # GET /api/admin/clear-cache
│
├── frontend/                            # Layer 5 : Interface Utilisateur Web
│   └── src/
│       ├── App.jsx                      # Application React complète (cartes, filtres, modale)
│       └── index.css                    # Design system sombre et épuré
├── notebooks/                           # Documentation et analyses approfondies
└── requirements.txt                     # Dépendances applicatives
```


In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15)
ax.set_ylim(0, 9.5)
ax.axis('off')
fig.patch.set_facecolor('#f8faff')

layers = [
    ("COUCHE 0 — Données Sources Brutes", 0.3, 8.0, 14.4, 0.9, '#e8f4fd', PRIMARY, ["data/raw/*.xlsx", "data/processed/*.csv"]),
    ("COUCHE 1 — Ingestion & Nettoyage (Data Layer)", 0.3, 6.6, 14.4, 0.9, '#f0fdf4', SUCCESS, ["src/data/loader.py", "src/data/cleaner.py"]),
    ("COUCHE 2 — Machine Learning & Features", 0.3, 5.2, 14.4, 0.9, '#fefce8', WARNING, ["feature_engineering.py", "target_builder.py", "train_classifier.py", "train_regressor.py"]),
    ("COUCHE 3 — Services Métier & Explicabilité", 0.3, 3.8, 14.4, 0.9, '#fff1f2', DANGER, ["recommendation.py", "explanation.py", "deep_context.py", "feedback.py"]),
    ("COUCHE 4 — API REST FastAPI", 0.3, 2.4, 14.4, 0.9, '#f5f3ff', '#7c3aed', ["main.py", "schemas.py", "routes/recommend.py", "routes/clients.py", "routes/feedback.py"]),
    ("COUCHE 5 — Interface Commerciale (React SPA)", 0.3, 1.0, 14.4, 0.9, '#ecfdf5', '#059669', ["frontend/src/App.jsx", "frontend/src/index.css"]),
]

for (label, x, y, w, h, fc, ec, files) in layers:
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                           facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 0.2, y + h - 0.2, label, va='top', ha='left',
            fontsize=9, fontweight='bold', color=ec)
    for j, fname in enumerate(files):
        ax.text(x + 0.3 + j * 3.4, y + 0.2, f"• {fname}", va='bottom', ha='left',
                fontsize=7.8, color=INK)

ax.set_title("Architecture en Couches et Dépendances du Projet SalesTeam AI", fontsize=13, fontweight='bold', color=INK, pad=12)
plt.tight_layout()
plt.savefig('../notebooks/figures/02_layers.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 3 — Architecture Logicielle & Diagrammes UML


In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlim(0, 15)
ax.set_ylim(0, 9)
ax.axis('off')
fig.patch.set_facecolor('#fafbff')

actors = [
    ("Commercial\n(React)", 1.2, PRIMARY),
    ("API\nFastAPI", 3.8, '#7c3aed'),
    ("Service\nrecommendation", 6.5, WARNING),
    ("Classifier\nXGBoost", 9.2, DANGER),
    ("Regressor\nXGBoost", 11.8, SUCCESS),
    ("Explicabilité\nexplanation.py", 14.0, MUTED),
]

for (label, x, color) in actors:
    rect = FancyBboxPatch((x-0.9, 7.7), 1.8, 0.9, boxstyle="round,pad=0.1",
                           facecolor=color, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 8.15, label, ha='center', va='center', fontsize=7.5,
            fontweight='bold', color='white')
    ax.plot([x, x], [7.7, 0.4], color=color, linewidth=0.9, linestyle='--', alpha=0.45)

arrows = [
    (1.2, 3.8, 7.2, "1. POST /api/recommend {client_id, visit_date}"),
    (3.8, 6.5, 6.5, "2. recommend(request, _skip_llm=True)"),
    (6.5, 9.2, 5.8, "3. predict_proba(X_clf [9 features])"),
    (9.2, 6.5, 5.1, "4. Probabilités d'achat P(achat)"),
    (6.5, 11.8, 4.4, "5. predict(X_reg [10 features]) [si CV <= 1.0]"),
    (11.8, 6.5, 3.7, "6. Quantités brutes prédites"),
    (6.5, 14.0, 3.0, "7. _rule_based_explanation() (gratuit, rapide)"),
    (14.0, 6.5, 2.3, "8. Textes d'explication courts"),
    (6.5, 3.8, 1.6, "9. RecommendResponse structurée"),
    (3.8, 1.2, 0.9, "10. JSON affiché dans les cartes d'urgence"),
]

for (x1, x2, y, label) in arrows:
    direction = 1 if x2 > x1 else -1
    ax.annotate("", xy=(x2 - direction*0.7, y), xytext=(x1 + direction*0.7, y),
                arrowprops=dict(arrowstyle="->", color=INK, lw=1.2))
    ax.text((x1+x2)/2, y+0.12, label, ha='center', va='bottom',
            fontsize=7.2, color=INK, style='italic')

ax.set_title("Diagramme de Séquence — Requête de Recommandation Complète", fontsize=12, fontweight='bold', color=INK, y=0.98)
plt.tight_layout()
plt.savefig('../notebooks/figures/03_sequence.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 4 — Couche Données : Ingestion et Nettoyage

### 4.1 Signatures et Rôles des Fonctions (`loader.py` & `cleaner.py`)
- `load_factures(filepath)` : Charge l'entête des factures, standardise `No_` en `code_facture`, parse les dates d'enregistrement.
- `load_lignes(filepath)` : Charge le détail des commandes, isole les articles et les quantités vendues.
- `load_clients(filepath)` : Charge les coordonnées GPS des points de vente.
- `build_main_table(lignes, factures, clients)` : Réalise la fusion relationnelle.

### 4.2 Le Problème Critique de la Clé Composée `(code_facture + societe)`
Dans le système ERP Navision d'origine, plusieurs filiales (*LSAT*, *NEWTECH*, *ONETEL*) partagent des séquences de numérotation similaires pour leurs factures (`No_`). 
Une jointure naïve sur `code_facture` seul générait des **produits croisés incohérents** entre filiales. 
La solution stricte appliquée dans `loader.py` :
```python
df = lignes.merge(
    factures,
    left_on=["code_facture", "societe_ligne"],
    right_on=["code_facture", "societe"],
    how="inner"
)
```


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

for a in ax:
    a.set_xlim(0, 10)
    a.set_ylim(0, 6)
    a.axis('off')

ax[0].set_title("❌ AVANT : Jointure naïve sur code_facture\n(Création de faux doublons inter-sociétés)", fontsize=10, fontweight='bold', color=DANGER)
ax[0].add_patch(FancyBboxPatch((0.5, 1.2), 9, 3.8, boxstyle="round,pad=0.1", facecolor='#fef2f2', edgecolor=DANGER, linewidth=1.5))
ax[0].text(5, 3.5, "Facture F-1001 (LSAT)  x  Facture F-1001 (NEWTECH)\n= Lignes fusionnées à tort (Erreur d'attribution client)",
           ha='center', va='center', fontsize=8.5, color=INK)

ax[1].set_title("✅ APRÈS : Jointure sur (code_facture, societe)\n(Isolation stricte et intégrité relationnelle)", fontsize=10, fontweight='bold', color=SUCCESS)
ax[1].add_patch(FancyBboxPatch((0.5, 1.2), 9, 3.8, boxstyle="round,pad=0.1", facecolor='#f0fdf4', edgecolor=SUCCESS, linewidth=1.5))
ax[1].text(5, 3.5, "Jointure composite :\n[code_facture + societe_ligne] == [code_facture + societe]\n= Intégrité 100% garantie sur les 78 530 lignes",
           ha='center', va='center', fontsize=8.5, color=INK)

plt.tight_layout()
plt.savefig('../notebooks/figures/04_join_problem.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 5 — Feature Engineering (Ingénierie des Caractéristiques)

### 5.1 & 5.2 Les 5 Groupes de Caractéristiques Métier
1. **Groupe 1 — Dynamique d'Achat Client/Article** :
   - `frequency` : Nombre total de commandes passées par le client pour cet article.
   - `avg_qty`, `median_qty`, `std_qty`, `min_qty`, `max_qty`, `last_qty` : Statistiques de volume.
   - `recency_days` : Nombre de jours écoulés depuis le dernier achat.
   - `avg_delay_days` : Intervalle moyen entre deux commandes successives.
   - `recency_relative` : $\frac{\text{recency\_days}}{\text{avg\_delay\_days}}$ (Indicateur clé : $> 1.0$ = réapprovisionnement en retard).
   - `trend` : Évolution des volumes récents vs anciens, bornée sur $[-0.9, 3.0]$.
2. **Groupe 2 — Saisonnalité Catégorielle** :
   - `cat_quarterly_coef` : Indice saisonnier calculé au trimestre par catégorie.
3. **Groupe 3 — Géolocalisation** :
   - `latitude`, `longitude`, `has_gps`.
4. **Groupe 4 — Cycle de Vie Article** :
   - `is_new_product` (lancé il y a $\le 90$ jours), `is_bulk_product`, `nb_clients`.
5. **Groupe 5 — Profil Global Client** :
   - `client_total_products`, `client_total_invoices`, `client_avg_basket_size`.

### 5.3 Focus Saisonnalité : Calcul et Propagation
L'indice saisonnier est calculé par `build_categorical_quarterly_index()` en éliminant l'anomalie de volume du 13 au 20 octobre 2024 (commandes exceptionnelles de déstockage) :
$$\text{coef}[\text{cat}][Q] = \frac{\text{Moyenne}(\text{quantité} \mid \text{cat}, Q)}{\text{Moyenne}(\text{quantité} \mid \text{cat}, \text{annuelle})}$$
Ce coefficient est utilisé de manière cohérente à deux niveaux :
1. Dans le **classifieur** comme feature prédictive de l'appétence saisonnière.
2. Dans le **régresseur** pour ajuster les volumes recommandés à la hausse ou à la baisse.


In [ ]:
fm = pd.read_csv('../data/processed/feature_matrix.csv')
numeric_cols = ['avg_qty','median_qty','std_qty','min_qty','max_qty','frequency',
                'recency_days','avg_delay_days','recency_relative','trend','cat_quarterly_coef']
corr = fm[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdYlBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.75)

ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right', fontsize=8.5)
ax.set_yticklabels(numeric_cols, fontsize=8.5)

for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.55 else INK
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

ax.set_title("Matrice de Corrélation des Features (feature_matrix.csv)", fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('../notebooks/figures/05_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 6 — Construction de la Cible (Target Builder)

### 6.1 Pourquoi la Formulation Visit-Level est Cruciale
Dans les approches naïves de recommandation, labelliser tous les achats passés comme positifs et les non-achats comme négatifs crée une **fuite temporelle majeure** : la feature `frequency` est nulle pour tous les négatifs et strictement positive pour les positifs. Le modèle apprend alors une règle triviale (`if frequency > 0`) sans pouvoir discriminer le moment opportun d'achat.

### 6.2 L'Approche Visit-Level
Pour chaque facture historique d'un client à une date $T$ :
- L'historique utilisé pour calculer les features s'arrête **strictement avant** $T$.
- $\text{target\_bought} = 1$ si le produit a été commandé sur la facture à $T$.
- $\text{target\_bought} = 0$ si le produit faisait partie des articles connus du client mais n'a pas été commandé lors de cette visite.
- Seuil minimal de recul : `MIN_HISTORY_ORDERS = 3` (minimum 3 commandes préalables pour qualifier un client).


In [ ]:
ts = pd.read_csv('../data/processed/training_set.csv')
pos_count = ts['target_bought'].sum()
total_count = len(ts)
pos_rate = ts['target_bought'].mean() * 100

print("=== STATISTIQUES DU DATASET D'ENTRAÎNEMENT (training_set.csv) ===")
print(f"Nombre total d'exemples (visite x article) : {total_count:,}")
print(f"Achats effectifs (target = 1)              : {pos_count:,} ({pos_rate:.1f}%)")
print(f"Non-achats lors de la visite (target = 0)  : {total_count - pos_count:,} ({100 - pos_rate:.1f}%)")
print(f"Déséquilibre de classe                     : 1 positif pour {(total_count - pos_count)//pos_count} négatifs")
print(f"Plage temporelle des visites               : {ts['visit_date'].min()[:10]} au {ts['visit_date'].max()[:10]}")


# Section 7 — Modèle Classifieur XGBoost


In [ ]:
with open('../src/models/classifier_lsat_metadata.json') as f:
    clf_meta = json.load(f)

clf = joblib.load('../src/models/classifier_lsat.joblib')
feat_cols = clf_meta['feature_columns']
importances = clf.feature_importances_

sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = [DANGER if importances[i] == max(importances) else PRIMARY for i in sorted_idx]
bars = ax.barh([feat_cols[i] for i in sorted_idx], importances[sorted_idx], color=colors, height=0.6)
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f} ({val*100:.1f}%)',
            va='center', fontsize=8.5, color=INK)

ax.set_title(f"Importance des Features — Classifieur XGBoost (ROC-AUC = {clf_meta['metrics']['roc_auc']:.4f})", fontsize=11, fontweight='bold')
ax.set_xlabel("Gain Relatif (Importance)", fontsize=9.5)
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/figures/07_clf_importance.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 8 — Modèle Régresseur XGBoost (IA vs Baseline)


In [ ]:
with open('../src/models/regressor_lsat_metadata.json') as f:
    reg_meta = json.load(f)

reg = joblib.load('../src/models/regressor_lsat.joblib')
reg_cols = reg_meta['feature_columns']
reg_imp = reg.feature_importances_
m = reg_meta['metrics_rounded']

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Plot 1: Importances
sorted_idx = np.argsort(reg_imp)
axes[0].barh([reg_cols[i] for i in sorted_idx], reg_imp[sorted_idx], color=SUCCESS, height=0.6)
for bar, val in zip(axes[0].patches, reg_imp[sorted_idx]):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val*100:.1f}%', va='center', fontsize=8)
axes[0].set_title("Importance Features — Régresseur", fontsize=10.5, fontweight='bold')
axes[0].set_facecolor('#f8f9fa')
axes[0].grid(axis='x', alpha=0.3)

# Plot 2: Comparaison MAE
bars2 = axes[1].bar(['XGBoost IA', 'Baseline (avg_qty)'], [m['mae'], m['mae_baseline']], color=[SUCCESS, MUTED], width=0.45)
for bar, val in zip(bars2, [m['mae'], m['mae_baseline']]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15, f'{val:.2f} unités', ha='center', fontsize=9.5, fontweight='bold')
axes[1].set_ylim(0, max([m['mae'], m['mae_baseline']]) * 1.3)
axes[1].set_title(f"Erreur Moyenne Absolue (Gain IA = +{m['improvement_mae_pct']:.1f}%)", fontsize=10.5, fontweight='bold')
axes[1].set_facecolor('#f8f9fa')
axes[1].grid(axis='y', alpha=0.3)

fig.patch.set_facecolor('#f8f9fa')
plt.tight_layout()
plt.savefig('../notebooks/figures/08_regressor.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 9 — Pipeline Complet de Recommandation & Classement


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
fig.patch.set_facecolor('#fafbff')

def draw_node(x, y, w, h, text, color):
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1", facecolor=color, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=8, color='white', fontweight='bold')

draw_node(0.5, 2.0, 2.6, 1.0, "Candidat Produit\n(Client, Article)", PRIMARY)
draw_node(3.8, 2.0, 2.6, 1.0, "Calcul CV\n(std_qty / avg_qty)", MUTED)
draw_node(7.2, 3.2, 3.0, 1.0, "CV > 1.0 (Irrégulier)\n-> avg_qty (Historique)", WARNING)
draw_node(7.2, 0.8, 3.0, 1.0, "CV <= 1.0 (Régulier)\n-> XGBoost Regressor", SUCCESS)
draw_node(10.8, 2.0, 2.8, 1.0, "_clamp_prediction()\n[qty_min, qty_sugg, qty_max]", DANGER)

ax.annotate("", xy=(3.8, 2.5), xytext=(3.1, 2.5), arrowprops=dict(arrowstyle="->", lw=1.5, color=INK))
ax.annotate("", xy=(7.2, 3.7), xytext=(6.4, 2.8), arrowprops=dict(arrowstyle="->", lw=1.5, color=INK))
ax.annotate("", xy=(7.2, 1.3), xytext=(6.4, 2.2), arrowprops=dict(arrowstyle="->", lw=1.5, color=INK))
ax.annotate("", xy=(10.8, 2.7), xytext=(10.2, 3.7), arrowprops=dict(arrowstyle="->", lw=1.5, color=INK))
ax.annotate("", xy=(10.8, 2.3), xytext=(10.2, 1.3), arrowprops=dict(arrowstyle="->", lw=1.5, color=INK))

ax.set_title("Arbre de Décision de la Prédiction de Quantité (CV-Switch + Bornage)", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/figures/09_qty_flow.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 10 — L'API REST et l'Élucidation des 569 Clients


In [ ]:
cmd = pd.read_csv('../data/processed/commandes_clean.csv')
fm  = pd.read_csv('../data/processed/feature_matrix.csv')
ts  = pd.read_csv('../data/processed/training_set.csv')
gps = pd.read_csv('../data/processed/gps_clean.csv')

n_cmd = cmd['code_client'].nunique()
n_fm  = fm['code_client'].nunique()
n_ts  = ts['code_client'].nunique()
n_gps = gps['code_client'].nunique()

print(f"1. Base Brute (commandes_clean.csv) : {n_cmd} clients uniques")
print(f"2. Matrice Features (feature_matrix.csv) : {n_fm} clients uniques")
print(f"3. Dataset Entraînement (training_set.csv) : {n_ts} clients uniques")
print(f"4. Réponse API (/api/clients) : {n_ts} clients")
print(f"-> Différence exacte : {n_cmd - n_ts} clients écartés par MIN_HISTORY_ORDERS = 3")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
fig.patch.set_facecolor('#fafbff')

funnel_steps = [
    (737, "Clients avec factures historiques (commandes_clean.csv)", PRIMARY, 4.8, 6.0),
    (737, "Clients avec matrice de features calculée (feature_matrix.csv)", MUTED, 3.6, 5.0),
    (569, "Clients éligibles avec >= 3 commandes (training_set.csv / API)", SUCCESS, 2.4, 4.0),
    (168, "Clients exclus (< 3 commandes = historique insuffisant)", DANGER, 1.2, 3.2),
]

for count, desc, color, y, w in funnel_steps:
    x_start = (10 - w) / 2
    rect = FancyBboxPatch((x_start, y-0.4), w, 0.8, boxstyle="round,pad=0.08", facecolor=color, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(5, y, f"{count} clients : {desc}", ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.set_title("Entonnoir de Filtrage et Résolution de la Discordance Client (737 -> 569)", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/figures/10_client_funnel.png', dpi=150, bbox_inches='tight')
plt.show()


# Section 11 — Couche Explication & Traitement du Langage Naturel

### 11.1 Architecture à Double Régime
1. **Régime Rapide (Cartes de Recommandation)** :
   - `_skip_llm = True` forcé sur la route `/api/recommend`.
   - Utilise le générateur déterministe en Python (`_rule_based_explanation`).
   - Latence $< 10$ millisecondes, zéro coût de token.
2. **Régime Approfondi (Modale d'Explication Détaillée)** :
   - Déclenché **uniquement au clic utilisateur** sur `/api/explain-detailed`.
   - Assemblage du contexte structuré via `deep_context.py`.
   - Appel LLM avec circuit-breaker et bascule automatique sur `_rule_based_detailed_explanation` en cas de défaillance réseau ou d'indisponibilité de quota.


# Section 12 — Architecture Frontend (React SPA)

L'application client (`frontend/src/App.jsx`) est une SPA réactive sans dépendance lourde :
- **Gestion d'État** : Sélection du client, date de visite dynamique, gestion de l'historique et affichage modale.
- **Organisation Visuelle** : Deux sections bien identifiées (*Urgent* pour les réassorts critiques en rouge, *Recommandé* en vert).
- **Modale Redimensionnable** : Panneau d'audit dynamique (`resize: both`) affichant l'explication en 4 parties claires sans jargon technique.


# Section 13 — Guide de Lecture pour un Nouvel Ingénieur

Pour prendre en main efficacement la base de code, suivre cet ordre de lecture précis :
1. `src/data/loader.py` & `src/data/cleaner.py` : Comprendre les règles de jointure et de nettoyage.
2. `src/features/feature_engineering.py` : Maîtriser le calcul des 27 variables descriptives.
3. `src/models/target_builder.py` : Comprendre la formulation anti-fuite visit-level.
4. `src/models/train_classifier.py` & `train_regressor.py` : Examiner les métriques d'apprentissage.
5. `src/services/recommendation.py` : Comprendre le moteur d'orchestration et le scoring final.
6. `src/services/deep_context.py` & `explanation.py` : Analyser la production des justifications métier.
7. `src/api/main.py` & `routes/` : Examiner les contrats d'interface FastAPI.
8. `frontend/src/App.jsx` : Visualiser l'expérience utilisateur finale.


# Section 14 — Feuille de Route vers la Production

| Priorité | Chantier Technique | Description & Actions Requises |
|---|---|---|
| 🔴 **Critique** | **Sécurisation & Authentification** | Ajouter un middleware JWT sur FastAPI pour restreindre l'accès aux commerciaux habilités. |
| 🔴 **Critique** | **Couverture de Tests (CI)** | Mettre en place `pytest` pour tester le chargement des modèles et la cohérence de l'API. |
| 🔴 **Critique** | **Conteneurisation Docker** | Écrire `Dockerfile` et `docker-compose.yml` pour le déploiement sur serveur d'entreprise. |
| 🟠 **Important** | **Boucle de Rétroaction Active** | Intégrer les retours enregistrés dans `data/feedback/` dans le ré-entraînement périodique. |
| 🟡 **Confort** | **Monitoring & Télémétrie** | Ajouter un suivi Prometheus/Grafana sur les temps de réponse et la distribution des scores. |
